# SQL Practice - Day 10

## Mixed Interview Practice

Topics: CASE WHEN, Window Functions, Self Join, INNER/LEFT JOIN, Subqueries, Correlated Subqueries, CTE + Windows.

10 mixed questions for 2–3 year Data Analyst interviews.

## Q1

**Task:** Using CASE WHEN, classify employees as High (>=80000), Medium (50000–79999), Low (<50000).

In [1]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['A','B','C','D','E'],'salary':[90000,75000,45000,80000,60000]})
df

,emp_id,name,salary
0,1,A,90000
1,2,B,75000
2,3,C,45000
3,4,D,80000
4,5,E,60000


In [10]:
import pandasql
from pandasql import sqldf
sqldf("""
select emp_id , name , salary,
case
    when salary >= 80000 then "High"
    when salary >=50000 and salary <= 79999 then "Medium"
    else "Low"
end as "Buket"
from df


""")

,emp_id,name,salary,Buket
0,1,A,90000,High
1,2,B,75000,Medium
2,3,C,45000,Low
3,4,D,80000,High
4,5,E,60000,Medium


## Q2

**Task:** Find the top 2 highest-paid employees in each department using DENSE_RANK().

In [11]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6,7],'name':['A','B','C','D','E','F','G'],'department':['HR','HR','HR','IT','IT','Sales','Sales'],'salary':[60000,80000,80000,70000,90000,50000,70000]})
df

,emp_id,name,department,salary
0,1,A,HR,60000
1,2,B,HR,80000
2,3,C,HR,80000
3,4,D,IT,70000
4,5,E,IT,90000
5,6,F,Sales,50000
6,7,G,Sales,70000


In [14]:
sqldf("""
select *
from (select * , 
        dense_rank()
        over(partition by department order by salary desc) as rn 
        from df) t
where rn = 2

""")

,emp_id,name,department,salary,rn
0,1,A,HR,60000,2
1,4,D,IT,70000,2
2,6,F,Sales,50000,2


## Q3

**Task:** Using a SELF JOIN, find employees whose salary is greater than their manager's salary.

In [15]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['ManagerA','EmpB','EmpC','ManagerD','EmpE'],'manager_id':[None,1,1,None,4],'salary':[80000,90000,70000,70000,75000]})
df

,emp_id,name,manager_id,salary
0,1,ManagerA,NaN,80000
1,2,EmpB,1.0,90000
2,3,EmpC,1.0,70000
3,4,ManagerD,NaN,70000
4,5,EmpE,4.0,75000


In [21]:
sqldf("""
select e.name as Emp_name  , e.salary as emp_salart , m.name as manager_name  ,m.salary as manager_salary
from df e
join df m on e.manager_id = m.emp_id
where e.salary > m.salary
""")

,Emp_name,emp_salart,manager_name,manager_salary
0,EmpB,90000,ManagerA,80000
1,EmpE,75000,ManagerD,70000


## Q4

**Task:** Using a LEFT JOIN, find customers who never placed an order.

In [22]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'name':['A','B','C','D']})
df1=pd.DataFrame({'order_id':[1,2,3],'customer_id':[101,103,103]})
print(df)
print(df1)

   customer_id name
0          101    A
1          102    B
2          103    C
3          104    D
   order_id  customer_id
0         1          101
1         2          103
2         3          103


In [23]:
sqldf("""
select c.name
from df c
left join df1 o on c.customer_id = o.customer_id
where o.order_id is null


""")

,name
0,B
1,D


## Q5

**Task:** Find products priced above their category average using a correlated subquery.

In [59]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,200,150,300,500,400]})
df

,product_id,category,price
0,1,A,100
1,2,A,200
2,3,A,150
3,4,B,300
4,5,B,500
5,6,B,400


In [25]:
sqldf("""
select category , price
from df a
where price > (select avg(price)
                from df b
                where a.category = b.category)


""")

,category,price
0,A,200
1,B,500


## Q6

**Task:** Find customers whose total spending is greater than the average customer spending using a CTE.

In [62]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104],'amount':[100,300,500,700,200,400]})
df

,customer_id,amount
0,101,100
1,101,300
2,102,500
3,103,700
4,104,200
5,104,400


In [63]:
sqldf("""
with my_cte as (
    select customer_id , sum(amount) as total_spending
    from df
    group  by customer_id  
)
select * from my_cte
where total_spending > (select avg(total_spending)
                        from my_cte)
""")

,customer_id,total_spending
0,103,700
1,104,600


## Q7

**Task:** For each employee, show salary and department average salary using a window function. Add CASE: Above Average / Below Average.

In [64]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E'],'department':['HR','HR','IT','IT','IT'],'salary':[50000,70000,60000,90000,65000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,70000
2,C,IT,60000
3,D,IT,90000
4,E,IT,65000


In [65]:
sqldf("""
SELECT
    name,
    department,
    salary,
    AVG(salary) OVER(PARTITION BY department) AS dept_avg_salary,
    CASE
        WHEN salary > AVG(salary) OVER(PARTITION BY department)
            THEN 'Above Average'
        WHEN salary < AVG(salary) OVER(PARTITION BY department)
            THEN 'Below Average'
        ELSE 'Equal to Average'
    END AS salary_status
FROM df
""")

,name,department,salary,dept_avg_salary,salary_status
0,A,HR,50000,60000.000000,Below Average
1,B,HR,70000,60000.000000,Above Average
2,C,IT,60000,71666.666667,Below Average
3,D,IT,90000,71666.666667,Above Average
4,E,IT,65000,71666.666667,Below Average


## Q8

**Task:** Find the latest order for every customer using ROW_NUMBER(), then return only the latest order.

In [66]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,102,103,103],'order_date':['2025-01-01','2025-03-01','2025-02-01','2025-01-15','2025-01-10','2025-04-01'],'amount':[100,300,500,200,700,800]})
df

,order_id,customer_id,order_date,amount
0,1,101,2025-01-01,100
1,2,101,2025-03-01,300
2,3,102,2025-02-01,500
3,4,102,2025-01-15,200
4,5,103,2025-01-10,700
5,6,103,2025-04-01,800


In [67]:
sqldf("""
WITH ranked_orders AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY order_date DESC
        ) AS rn
    FROM df
)
SELECT
    order_id,
    customer_id,
    order_date,
    amount
FROM ranked_orders
WHERE rn = 1
""")

,order_id,customer_id,order_date,amount
0,2,101,2025-03-01,300
1,3,102,2025-02-01,500
2,6,103,2025-04-01,800


## Q9

**Task:** Find departments whose average salary is greater than the overall company average salary.

In [69]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[50000,60000,70000,80000,90000,85000]})
df

,emp_id,department,salary
0,1,HR,50000
1,2,HR,60000
2,3,IT,70000
3,4,IT,80000
4,5,Sales,90000
5,6,Sales,85000


In [70]:
sqldf("""
select department , avg(salary) as avg_department_salary
from df
group by department
having avg(salary) > (select avg(salary) from df)


""")

,department,avg_department_salary
0,IT,75000.0
1,Sales,87500.0


## Q10

**Task:** Find the second-highest total-spending customer using aggregation + DENSE_RANK().

In [72]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104],'amount':[100,300,500,700,200,500]})
df

,customer_id,amount
0,101,100
1,101,300
2,102,500
3,103,700
4,104,200
5,104,500


In [75]:
sqldf("""
with my_cte as (
select  customer_id , sum(amount) as total_spending, 
        dense_rank()
        over(order by sum(amount) desc) as rn
from df
group by customer_id
)
select * from my_cte
where rn = 2
""")

,customer_id,total_spending,rn
0,102,500,2
